# MLFlow Experiments & Tracking

This notebook demonstrates advanced MLFlow features for tracking and comparing agent experiments.

## Topics Covered

1. Setting up experiments
2. Tracking parameters and metrics
3. Logging artifacts
4. Comparing runs
5. Best practices for agent experiments


In [ ]:
# Setup
import time
import json
from agentic_assistants import AgenticConfig, OllamaManager
from agentic_assistants.core import MLFlowTracker
from agentic_assistants.utils.logging import setup_logging

setup_logging(level="INFO")

config = AgenticConfig()
tracker = MLFlowTracker(config)

print(f"MLFlow enabled: {tracker.enabled}")
print(f"Tracking URI: {config.mlflow.tracking_uri}")
print(f"Default experiment: {config.mlflow.experiment_name}")


## Creating Experiments

Experiments group related runs together for comparison:


In [ ]:
# Set a specific experiment
tracker.experiment_name = "agent-comparison-study"

# Simulated agent configurations to compare
agent_configs = [
    {"model": "llama3.2", "temperature": 0.7, "max_tokens": 512},
    {"model": "llama3.2", "temperature": 0.3, "max_tokens": 512},
    {"model": "llama3.2", "temperature": 0.7, "max_tokens": 256},
]

print(f"Will compare {len(agent_configs)} configurations")


## Running Multiple Experiments

Let's simulate running multiple agent configurations:


In [ ]:
import random

for i, agent_config in enumerate(agent_configs):
    run_name = f"config-{i+1}-temp{agent_config['temperature']}"
    
    with tracker.start_run(run_name=run_name, tags={"experiment_type": "comparison"}) as run:
        # Log all parameters
        tracker.log_params(agent_config)
        tracker.log_param("config_id", i + 1)
        
        # Simulate agent execution with varying results
        start_time = time.time()
        time.sleep(0.1 + random.random() * 0.2)  # Simulated work
        duration = time.time() - start_time
        
        # Simulate metrics (in real usage, these would come from actual runs)
        quality_score = 0.7 + random.random() * 0.25
        response_length = agent_config["max_tokens"] * (0.6 + random.random() * 0.4)
        
        # Log metrics
        tracker.log_metrics({
            "duration_seconds": duration,
            "quality_score": quality_score,
            "response_length": int(response_length),
            "tokens_per_second": response_length / duration,
        })
        
        # Log a sample output as artifact
        sample_output = f"Sample output for config {i+1} with temperature {agent_config['temperature']}"
        tracker.log_text(sample_output, f"outputs/config_{i+1}.txt")
        
        # Log config as JSON artifact
        tracker.log_dict(agent_config, f"configs/config_{i+1}.json")
        
        print(f"✓ Completed run: {run_name} (quality: {quality_score:.2f})")


## Decorator-Based Tracking

You can also use decorators for automatic tracking:


In [ ]:
from agentic_assistants.core import track_experiment

@track_experiment("decorated-experiments", log_args=True)
def simulated_agent_task(model: str, temperature: float, query: str):
    """Simulated agent task that's automatically tracked."""
    time.sleep(0.1)  # Simulated work
    return f"Response from {model} at temp {temperature} for: {query}"

# Run the decorated function
result = simulated_agent_task(
    model="llama3.2",
    temperature=0.5,
    query="What is machine learning?"
)

print(f"Result: {result[:50]}...")


## Viewing Results

Open the MLFlow UI to compare runs:

```bash
# Start MLFlow server if not running
agentic mlflow start

# Open in browser
agentic mlflow ui
```

Or navigate to: http://localhost:5000

### In the UI you can:
- Compare runs side-by-side
- View parameter differences
- Plot metrics over time
- Download artifacts
- Search and filter runs


## Best Practices for Agent Experiments

### 1. Consistent Parameters
Always log the same parameters across runs:
- Model name and version
- Temperature and sampling settings
- Prompt templates (as artifacts)
- Agent configurations

### 2. Meaningful Metrics
Track metrics that help comparison:
- Response quality scores
- Latency/duration
- Token usage
- Error rates

### 3. Artifact Organization
Store related artifacts together:
- `prompts/` - Prompt templates
- `outputs/` - Agent outputs
- `configs/` - Configuration files
- `interactions/` - Conversation logs

### 4. Tags for Filtering
Use tags to categorize runs:
- `experiment_type`: comparison, baseline, production
- `agent_framework`: crewai, langgraph
- `task_type`: research, coding, analysis
